In [ ]:
import re
from types import SimpleNamespace

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', DEVICE)

In [ ]:
DATA_LINK = 'https://storage.yandexcloud.net/aiueducation/Content/base/l10/hh_fixed.csv'

gdown.download(DATA_LINK, None, quiet=True)

df = pd.read_csv('hh_fixed.csv', index_col=0)

print(df.shape)
df.head(3)

In [ ]:
column_names = {
    'sex_age': 'Пол, возраст',
    'salary': 'ЗП',
    'position_target': 'Ищет работу на должность:',
    'position_prev': 'Последеняя/нынешняя должность',
    'city': 'Город',
    'employment': 'Занятость',
    'schedule': 'График',
    'experience': 'Опыт (двойное нажатие для полной версии)',
    'education': 'Образование и ВУЗ',
    'updated': 'Обновление резюме',
}

COL_SEX_AGE = df.columns.get_loc(column_names['sex_age'])
COL_SALARY = df.columns.get_loc(column_names['salary'])
COL_POS_SEEK = df.columns.get_loc(column_names['position_target'])
COL_POS_PREV = df.columns.get_loc(column_names['position_prev'])
COL_CITY = df.columns.get_loc(column_names['city'])
COL_EMPL = df.columns.get_loc(column_names['employment'])
COL_SCHED = df.columns.get_loc(column_names['schedule'])
COL_EXP = df.columns.get_loc(column_names['experience'])
COL_EDU = df.columns.get_loc(column_names['education'])
COL_UPDATED = df.columns.get_loc(column_names['updated'])

In [ ]:
def normalize_text(value):
    if not isinstance(value, str):
        return ''
    return value.replace('\n', ' ').replace('\xa0', '').strip().lower()


def get_year(value):
    text = normalize_text(value)
    found = re.search(r'\d\d.\d\d.(\d{4})', text)
    return int(found[1]) if found else 0


def to_ohe(class_index, class_count):
    vector = np.zeros(class_count, dtype=np.float32)
    vector[class_index] = 1.0
    return vector

In [ ]:
currency_rate = {
    'usd': 65.,
    'kzt': 0.17,
    'грн': 2.6,
    'белруб': 30.5,
    'eur': 70.,
    'kgs': 0.9,
    'сум': 0.007,
    'azn': 37.5,
}

age_class = [0, [18, 23, 28, 33, 38, 43, 48, 53, 58, 63]]
experience_class = [0, [7, 13, 25, 37, 61, 97, 121, 157, 193, 241]]

city_class = [0, {
    'москва': 0,
    'санкт-петербург': 1,
    'новосибирск': 2,
    'екатеринбург': 2,
    'нижний новгород': 2,
    'казань': 2,
    'челябинск': 2,
    'омск': 2,
    'самара': 2,
    'ростов-на-дону': 2,
    'уфа': 2,
    'красноярск': 2,
    'пермь': 2,
    'воронеж': 2,
    'волгоград': 2,
    'прочие города': 3,
}]

employment_class = [0, {
    'стажировка': 0,
    'частичная занятость': 1,
    'проектная работа': 2,
    'полная занятость': 3,
}]

schedule_class = [0, {
    'гибкий график': 0,
    'полный день': 1,
    'сменный график': 2,
    'удаленная работа': 3,
}]

education_class = [0, {
    'высшее образование': 0,
    'higher education': 0,
    'среднее специальное': 1,
    'неоконченное высшее': 2,
    'среднее образование': 3,
}]

In [ ]:
def fill_class_count(class_info):
    labels = class_info[1]
    if isinstance(labels, list):
        class_info[0] = len(labels) + 1
    else:
        class_info[0] = max(labels.values()) + 1


for item in [age_class, experience_class, city_class, employment_class, schedule_class, education_class]:
    fill_class_count(item)

In [ ]:
def value_to_ohe(value, class_info):
    class_count, borders = class_info
    selected_class = class_count - 1

    for index, border in enumerate(borders):
        if value < border:
            selected_class = index
            break

    return to_ohe(selected_class, class_count)


def text_to_multi(value, class_info):
    text = normalize_text(value)
    result = np.zeros(class_info[0], dtype=np.float32)

    for marker, class_index in class_info[1].items():
        if marker in text:
            result[class_index] = 1.0

    return result

In [ ]:
BASE_YEAR = 2019


def split_sex_age(value):
    text = normalize_text(value)
    sex = 1.0 if 'муж' in text else 0.0

    found = re.search(r'\d{4}', text)
    age = BASE_YEAR - int(found[0]) if found else 0

    return sex, age


def age_to_ohe(age):
    return value_to_ohe(age, age_class)


def experience_to_ohe(months):
    return value_to_ohe(months, experience_class)

In [ ]:
def salary_to_thousands(value):
    text = normalize_text(value)
    found = re.search(r'\d+', text)

    if not found:
        return 0.0

    salary = float(found[0])
    for currency, rate in currency_rate.items():
        if currency in text:
            salary *= rate
            break

    return salary / 1000.0


def city_to_ohe(value):
    text = normalize_text(value)
    words = re.split(r'[ ,.:()?!]', text)

    class_index = city_class[0] - 1
    for word in words:
        candidate = city_class[1].get(word, -1)
        if candidate >= 0:
            class_index = candidate
            break

    return to_ohe(class_index, city_class[0])


def employment_to_multi(value):
    return text_to_multi(value, employment_class)


def schedule_to_multi(value):
    return text_to_multi(value, schedule_class)


def education_to_multi(value):
    result = text_to_multi(value, education_class)

    if result[2] > 0.0:
        result[0] = 0.0

    return result


def experience_months(value):
    text = normalize_text(value)

    years_found = re.search(r'(\d+)\s+(год.?|лет)', text)
    months_found = re.search(r'(\d+)\s+месяц', text)

    years = int(years_found[1]) if years_found else 0
    months = int(months_found[1]) if months_found else 0

    return years * 12 + months

In [ ]:
def row_to_sample(row):
    sex, age = split_sex_age(row[COL_SEX_AGE])
    months = experience_months(row[COL_EXP])

    features = np.hstack([
        np.array([sex], dtype=np.float32),
        age_to_ohe(age),
        city_to_ohe(row[COL_CITY]),
        employment_to_multi(row[COL_EMPL]),
        schedule_to_multi(row[COL_SCHED]),
        education_to_multi(row[COL_EDU]),
        experience_to_ohe(months),
    ]).astype(np.float32)

    target = np.array([salary_to_thousands(row[COL_SALARY])], dtype=np.float32)
    return features, target


def make_dataset(rows):
    x_values, y_values = [], []

    for row in rows:
        x, y = row_to_sample(row)
        if y[0] > 0:
            x_values.append(x)
            y_values.append(y)

    return np.array(x_values, dtype=np.float32), np.array(y_values, dtype=np.float32)

In [ ]:
x_train_01, y_train = make_dataset(df.values)

print(x_train_01.shape)
print(y_train.shape)

sample_index = 0
print(x_train_01[sample_index])
print(y_train[sample_index])

In [ ]:
def prepare_loaders(x_data, y_data, batch_size=32, val_part=0.2):
    x_tensor = torch.tensor(x_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data, dtype=torch.float32)

    count = len(x_tensor)
    indices = np.arange(count)
    np.random.shuffle(indices)

    val_count = int(count * val_part)
    val_indices = indices[:val_count]
    train_indices = indices[val_count:]

    train_ds = TensorDataset(x_tensor[train_indices], y_tensor[train_indices])
    val_ds = TensorDataset(x_tensor[val_indices], y_tensor[val_indices])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


def show_mae(history, title=''):
    data = history.history

    plt.figure(figsize=(8, 4))
    plt.plot(data['mae'], label='обучающая выборка')
    plt.plot(data['val_mae'], label='проверочная выборка')
    plt.xlabel('Эпоха')
    plt.ylabel('MAE')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def get_activation(name):
    activations = {
        'relu': nn.ReLU,
        'elu': nn.ELU,
        'linear': nn.Identity,
    }
    return activations.get(name, nn.ReLU)()


class SalaryRegressor(nn.Module):
    def __init__(self, input_size, layers_config):
        super().__init__()
        modules = []
        current_size = input_size

        for layer in layers_config:
            units = layer['units']
            activation = layer.get('activation', 'relu')

            modules.append(nn.Linear(current_size, units))

            if layer.get('batch_norm', False):
                modules.append(nn.BatchNorm1d(units))

            modules.append(get_activation(activation))

            dropout = layer.get('dropout', 0)
            if dropout:
                modules.append(nn.Dropout(dropout))

            current_size = units

        modules.append(nn.Linear(current_size, 1))
        self.net = nn.Sequential(*modules)

    def forward(self, x):
        return self.net(x)


def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_count = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            predictions = model(x_batch)
            loss = loss_fn(predictions, y_batch)
            mae = torch.abs(predictions - y_batch).sum().item()
            batch_count = len(x_batch)

            total_loss += loss.item() * batch_count
            total_mae += mae
            total_count += batch_count

    return total_loss / total_count, total_mae / total_count

In [ ]:
def train_model(title, layers_config, optimizer_cls, learning_rate, epochs=30, batch_size=32):
    train_loader, val_loader = prepare_loaders(x_train_01, y_train, batch_size=batch_size)

    model = SalaryRegressor(x_train_01.shape[1], layers_config).to(DEVICE)
    optimizer = optimizer_cls(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()

    history_data = {
        'loss': [],
        'mae': [],
        'val_loss': [],
        'val_mae': [],
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_mae = 0.0
        train_count = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            predictions = model(x_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            optimizer.step()

            batch_count = len(x_batch)
            train_loss += loss.item() * batch_count
            train_mae += torch.abs(predictions - y_batch).sum().item()
            train_count += batch_count

        epoch_loss = train_loss / train_count
        epoch_mae = train_mae / train_count
        val_loss, val_mae = evaluate_model(model, val_loader, loss_fn)

        history_data['loss'].append(epoch_loss)
        history_data['mae'].append(epoch_mae)
        history_data['val_loss'].append(val_loss)
        history_data['val_mae'].append(val_mae)

        print(
            f'Эпоха {epoch + 1:02d}/{epochs}: '
            f'MAE={epoch_mae:.4f}, val_MAE={val_mae:.4f}'
        )

    history = SimpleNamespace(history=history_data)
    print(f'{title}. Финальная MAE на проверке: {history.history["val_mae"][-1]}')
    show_mae(history, title)

    return model, history

In [ ]:
model_1, history_1 = train_model(
    title='Модель 1: простая полносвязная сеть',
    layers_config=[
        {'units': 64, 'activation': 'relu'},
        {'units': 32, 'activation': 'relu'},
    ],
    optimizer_cls=torch.optim.Adam,
    learning_rate=0.001,
    epochs=30,
    batch_size=32
)

In [ ]:
model_2, history_2 = train_model(
    title='Модель 2: глубокая сеть',
    layers_config=[
        {'units': 128, 'activation': 'relu'},
        {'units': 64, 'activation': 'relu'},
        {'units': 32, 'activation': 'relu'},
        {'units': 16, 'activation': 'relu'},
        {'units': 8, 'activation': 'relu'},
    ],
    optimizer_cls=torch.optim.Adam,
    learning_rate=0.001,
    epochs=30,
    batch_size=32
)

In [ ]:
model_3, history_3 = train_model(
    title='Модель 3: сеть с Dropout',
    layers_config=[
        {'units': 256, 'activation': 'relu', 'dropout': 0.3},
        {'units': 128, 'activation': 'relu', 'dropout': 0.3},
        {'units': 64, 'activation': 'relu'},
    ],
    optimizer_cls=torch.optim.Adam,
    learning_rate=0.001,
    epochs=30,
    batch_size=32
)

In [ ]:
model_4, history_4 = train_model(
    title='Модель 4: сеть с BatchNormalization',
    layers_config=[
        {'units': 128, 'activation': 'relu', 'batch_norm': True},
        {'units': 64, 'activation': 'relu', 'batch_norm': True},
    ],
    optimizer_cls=torch.optim.Adam,
    learning_rate=0.001,
    epochs=30,
    batch_size=32
)

In [ ]:
model_5, history_5 = train_model(
    title='Модель 5: широкая сеть',
    layers_config=[
        {'units': 512, 'activation': 'relu'},
        {'units': 256, 'activation': 'relu'},
    ],
    optimizer_cls=torch.optim.Adam,
    learning_rate=0.0005,
    epochs=30,
    batch_size=64
)

In [ ]:
model_6, history_6 = train_model(
    title='Модель 6: RMSprop и ELU',
    layers_config=[
        {'units': 128, 'activation': 'elu'},
        {'units': 64, 'activation': 'elu'},
        {'units': 32, 'activation': 'elu'},
    ],
    optimizer_cls=torch.optim.RMSprop,
    learning_rate=0.001,
    epochs=30,
    batch_size=32
)

In [ ]:
results = pd.DataFrame({
    'Модель': [
        'Модель 1',
        'Модель 2',
        'Модель 3',
        'Модель 4',
        'Модель 5',
        'Модель 6',
    ],
    'Финальная val_MAE': [
        history_1.history['val_mae'][-1],
        history_2.history['val_mae'][-1],
        history_3.history['val_mae'][-1],
        history_4.history['val_mae'][-1],
        history_5.history['val_mae'][-1],
        history_6.history['val_mae'][-1],
    ]
})

results